In [1]:
import re
from docling.document_converter import DocumentConverter
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

_converter = DocumentConverter()


def limpar(s):
    s = s.replace("**", "").replace("<br>", " ")
    return s.strip()


def extrair_texto_e_tabelas(caminho_pdf):
 
    result = _converter.convert(caminho_pdf)
    texto_md = result.document.export_to_markdown()
    linhas = texto_md.split("\n")

    texto_sem_tabelas = []
    documentos_tabelas = []
    buffer_tabela = []

    def processar_buffer_tabela():
        linhas_tabela = [l for l in buffer_tabela if not re.match(r"^\|[\s\-:|]+\|$", l.strip())]
        celulas = [[limpar(c) for c in l.strip().strip("|").split("|")] for l in linhas_tabela]
        if len(celulas) < 2:
            return
        cabecalho = celulas[0]
        for linha in celulas[1:]:
            partes = [
                f"{cabecalho[j]}: {linha[j]}"
                for j in range(min(len(cabecalho), len(linha)))
                if linha[j]
            ]
            if partes:
                frase = " | ".join(partes) + "."
                documentos_tabelas.append(Document(
                    page_content=frase,
                    metadata={"source": caminho_pdf, "tipo": "tabela"}
                ))

    for linha in linhas:
        if linha.strip().startswith("|"):
            buffer_tabela.append(linha)
        else:
            if buffer_tabela:
                processar_buffer_tabela()
                buffer_tabela = []
            texto_sem_tabelas.append(limpar(linha))

    if buffer_tabela:
        processar_buffer_tabela()

    documento_texto = Document(
        page_content="\n".join(texto_sem_tabelas),
        metadata={"source": caminho_pdf, "tipo": "texto"}
    )

    return documento_texto, documentos_tabelas





c:\Users\User\Desktop\voa-bank-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:

caminhos_pdfs = [
    r"data\documentos_fonte\01_politica_privacidade_protecao_dados.pdf",
    r"data\documentos_fonte\02_termos_condicoes_uso.pdf",
    r"data\documentos_fonte\03_faq_transacoes_limites.pdf",
    r"data\documentos_fonte\04_politica_seguranca_prevencao_fraudes.pdf",
    r"data\documentos_fonte\05_tarifas_comissoes.pdf",
]

documentos_texto = []
documentos_tabelas = []

for caminho in caminhos_pdfs:
    doc_texto, docs_tab = extrair_texto_e_tabelas(caminho)
    documentos_texto.append(doc_texto)
    documentos_tabelas.extend(docs_tab)

print(f"Documentos de texto: {len(documentos_texto)}")
print(f"Chunks de tabela (já prontos, um por linha): {len(documentos_tabelas)}")


[INFO] 2026-08-22 20:36:04,413 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-08-22 20:36:04,424 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-08-22 20:36:04,427 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.9.2/torch/PP-OCRv6/det/PP-OCRv6_det_small.pth
[INFO] 2026-08-22 20:36:08,684 [RapidOCR] download_file.py:82: Download size: 9.77MB
[INFO] 2026-08-22 20:36:09,339 [RapidOCR] download_file.py:95: Successfully saved to: C:\Users\User\Desktop\voa-bank-rag\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.pth
[INFO] 2026-08-22 20:36:09,341 [RapidOCR] main.py:50: Using C:\Users\User\Desktop\voa-bank-rag\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.pth
[INFO] 2026-08-22 20:36:09,623 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-08-22 20:36:09,624 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-08-22 20:36:09,626 [RapidOCR] download_file.py:68: I

Documentos de texto: 5
Chunks de tabela (já prontos, um por linha): 42


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=100)
splits_texto = text_splitter.split_documents(documentos_texto)

all_splits = splits_texto + documentos_tabelas

print(f"Chunks de texto: {len(splits_texto)}")
print(f"Chunks de tabela: {len(documentos_tabelas)}")
print(f"Total combinado: {len(all_splits)}")

Chunks de texto: 97
Chunks de tabela: 42
Total combinado: 139


In [7]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4119.55it/s]


In [8]:
from langchain_core.vectorstores import InMemoryVectorStore


vector_store = InMemoryVectorStore(embeddings)
vector_store.add_documents(documents=all_splits)
print(f"Indexed {len(all_splits)} chunks.")

Indexed 139 chunks.


In [9]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

retriever = vector_store.as_retriever(search_kwargs={"k": 6})

llm = ChatOllama(model="llama3.1", temperature=0)

prompt = ChatPromptTemplate.from_template("""
Você é um assistente do Voa Bank. Responda a pergunta do colaborador
usando APENAS o contexto abaixo.

Leia todos os trechos do contexto com atenção antes de responder.
Se o contexto afirmar ou negar algo relacionado à pergunta, isso é uma resposta válida
e você deve respondê-la normalmente — inclusive se a resposta for "não" ou uma negação.
Só diga que não encontrou a informação se o assunto da pergunta realmente não for
mencionado em nenhum trecho do contexto.

Quando o contexto descrever uma REGRA GERAL e também uma EXCEÇÃO a essa regra,
responda com base na regra geral primeiro, e mencione a exceção em seguida.
Não responda "sim" apenas porque uma exceção existe, se a regra geral for "não".

Sempre justifique sua resposta com uma frase curta baseada no contexto, para que o
colaborador possa verificar a fonte da informação. Não responda apenas "sim" ou "não"
sem explicação.

Base de informações:
{base}

Pergunta: {pergunta}

Resposta:
""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"base": retriever | format_docs, "pergunta": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
pergunta = input("Escreva sua pergunta: ")
resposta = rag_chain.invoke(pergunta)
print(resposta)

In [11]:
casos_de_teste = [
    {"id": "PRIV-01", "pergunta": "Qual é a legislação principal que fundamenta a política de privacidade do Voa Bank?",
     "resposta_esperada": "A Lei Geral de Proteção de Dados (Lei nº 13.709/2018 - LGPD)."},

    {"id": "PRIV-02", "pergunta": "Por quanto tempo o Voa Bank mantém o histórico de transações dos clientes?",
     "resposta_esperada": "10 anos, conforme exigência do Banco Central (BACEN)."},

    {"id": "PRIV-03", "pergunta": "Se eu quiser solicitar a exclusão dos meus dados pessoais, qual o prazo de resposta e por qual canal devo pedir?",
     "resposta_esperada": "Pelo aplicativo, em 'Configurações > Privacidade e Dados', ou pelo e-mail dpo@voabank.com.br. Prazo de até 15 dias úteis."},

    {"id": "PRIV-04", "pergunta": "Com quais birôs de crédito o Voa Bank pode compartilhar meus dados para análise de crédito?",
     "resposta_esperada": "Serasa Experian e Boa Vista SCPC."},

    {"id": "PRIV-05", "pergunta": "Qual tipo de criptografia o Voa Bank usa para dados em trânsito?",
     "resposta_esperada": "TLS 1.3."},

    {"id": "PRIV-06", "pergunta": "O Voa Bank vende dados de clientes para empresas de marketing?",
     "resposta_esperada": "Não. O documento afirma explicitamente que o Voa Bank não vende dados pessoais a terceiros para fins de marketing."},

    {"id": "TERMOS-01", "pergunta": "Qual a idade mínima para abrir uma conta no Voa Bank sem autorização de responsável legal?",
     "resposta_esperada": "18 anos. Entre 16 e 18 anos é permitido com autorização de responsável legal (conta Voa Jovem)."},

    {"id": "TERMOS-02", "pergunta": "O Voa Bank pode reduzir o limite do meu cartão de crédito sem aviso prévio?",
     "resposta_esperada": "Em geral não, exige aviso prévio de 15 dias. Exceção: suspeita de fraude ou inadimplência, onde a redução pode ser imediata."},

    {"id": "TERMOS-03", "pergunta": "O saldo em conta no Voa Bank é considerado depósito bancário?",
     "resposta_esperada": "Não. É segregado patrimonialmente conforme regulação do BACEN para instituições de pagamento."},

    {"id": "TERMOS-04", "pergunta": "Em quais situações o Voa Bank pode encerrar minha conta sem dar aviso prévio de 30 dias?",
     "resposta_esperada": "Suspeita fundamentada de fraude/lavagem de dinheiro/financiamento ao terrorismo; determinação legal ou regulatória; uso indevido comprovado."},

    {"id": "TERMOS-05", "pergunta": "Qual o número da central telefônica de atendimento do Voa Bank?",
     "resposta_esperada": "0800 555 0199 (ligação gratuita)."},

    {"id": "FAQ-01", "pergunta": "Qual o limite diário de Pix durante a noite (período noturno)?",
     "resposta_esperada": "R$ 1.000,00, válido das 20h às 6h, para novos usuários."},

    {"id": "FAQ-02", "pergunta": "Se eu aumentar meu limite de Pix para R$ 15.000,00, o aumento vale imediatamente?",
     "resposta_esperada": "Não. Aumentos acima de R$ 10.000,00 têm carência de 24 horas (Resolução BCB nº 150/2021)."},

    {"id": "FAQ-03", "pergunta": "É possível cancelar um Pix depois de enviado?",
     "resposta_esperada": "Não. Pode-se acionar o Mecanismo Especial de Devolução (MED) em até 80 dias em caso de fraude ou erro."},

    {"id": "FAQ-04", "pergunta": "Fiz uma compra internacional de US$ 100 no cartão de crédito. Quanto de IOF incide sobre essa transação?",
     "resposta_esperada": "IOF de 3,38% sobre o valor da transação internacional no cartão de crédito (não confundir com o IOF de 0,38% de remessa internacional/câmbio)."},

    {"id": "FAQ-05", "pergunta": "Até que horas um boleto precisa ser pago para ser processado no mesmo dia?",
     "resposta_esperada": "Até às 20h30 (horário de Brasília)."},

    {"id": "FAQ-06", "pergunta": "Existe algum limite para receber dinheiro via Pix?",
     "resposta_esperada": "Não há limite para recebimento de Pix. Limites se aplicam apenas a valores enviados."},

    {"id": "SEG-01", "pergunta": "O Voa Bank atende clientes por WhatsApp?",
     "resposta_esperada": "Não. Canal oficial é o chat do aplicativo ou 0800 555 0199."},

    {"id": "SEG-02", "pergunta": "Em quantos dias posso acionar o Mecanismo Especial de Devolução (MED) após uma fraude via Pix?",
     "resposta_esperada": "Até 80 dias após a transação."},

    {"id": "SEG-03", "pergunta": "O Voa Bank ressarce valores em qualquer caso de fraude?",
     "resposta_esperada": "Não em todos os casos. Ressarcimento integral só para falha de segurança da plataforma, não para engenharia social onde o usuário forneceu credenciais voluntariamente. Prazo de até 10 dias úteis."},

    {"id": "SEG-04", "pergunta": "O que é o golpe de 'SIM Swap' mencionado na política de segurança?",
     "resposta_esperada": "Transferência da linha telefônica da vítima para chip do criminoso, interceptando SMS de autenticação. Recomenda-se TOTP ou biometria em vez de SMS."},

    {"id": "SEG-05", "pergunta": "Quais tipos de autenticação de dois fatores são obrigatórios no Voa Bank?",
     "resposta_esperada": "2FA obrigatório para: alteração de senha, cadastro de nova chave Pix, aumento de limites e alteração de dados cadastrais."},

    {"id": "TAR-01", "pergunta": "Qual a mensalidade da conta Voa Bank Plus?",
     "resposta_esperada": "R$ 14,90 por mês (isenta com movimentação mínima de R$ 1.000/mês)."},

    {"id": "TAR-02", "pergunta": "Quanto custa uma TED na conta Voa Bank Light?",
     "resposta_esperada": "R$ 8,90 por transferência."},

    {"id": "TAR-03", "pergunta": "Comparando os três planos, qual oferece TED gratuita ilimitada?",
     "resposta_esperada": "Voa Bank Black. Plus oferece só até 5/mês grátis (depois R$ 6,90); Light cobra R$ 8,90 sempre."},

    {"id": "TAR-04", "pergunta": "Qual a taxa de juros rotativo do cartão de crédito no plano Voa Bank Black?",
     "resposta_esperada": "Até 11,90% ao mês."},

    {"id": "TAR-05", "pergunta": "Quanto custa sacar dinheiro em rede conveniada (Banco24Horas) com o cartão de débito?",
     "resposta_esperada": "R$ 6,50 por saque, sendo o primeiro saque do mês gratuito."},

    {"id": "TAR-06", "pergunta": "Qual a tarifa de manutenção da conta Voa Bank Light se eu não movimentar nenhum valor no mês?",
     "resposta_esperada": "Gratuita, sem exigência de movimentação mínima (diferente de Plus e Black)."},

    {"id": "MULTI-01", "pergunta": "Se eu suspeitar que minha conta foi usada por terceiros para receber e repassar dinheiro rapidamente (conta laranja), o que pode acontecer com minha conta?",
     "resposta_esperada": "Bloqueio preventivo (Política de Segurança) e possível encerramento sem aviso prévio de 30 dias, com comunicação às autoridades (Termos de Uso)."},

    {"id": "MULTI-02", "pergunta": "O Voa Bank pode bloquear minha conta por determinação de autoridades? Quais autoridades são mencionadas nos documentos?",
     "resposta_esperada": "Sim. BACEN, COAF, Receita Federal (Privacidade) e Poder Judiciário (Segurança)."},

    {"id": "TRAP-GERAL-01", "pergunta": "Qual o valor do CDI usado pelo Voa Bank para render o saldo em conta?",
     "resposta_esperada": "Não disponível nos documentos. Os Termos mencionam rendimento ao CDI, mas sem especificar percentual. Resposta correta é admitir que não sabe."},

    {"id": "TRAP-GERAL-02", "pergunta": "O Voa Bank tem agências físicas para atendimento presencial?",
     "resposta_esperada": "Não mencionado nos documentos. Só há referência a 'agência parceira' para emissão de extrato impresso — não confirma nem nega rede própria de agências."},
]

print(f"Total de casos de teste: {len(casos_de_teste)}\n")
print("=" * 100)

resultados = []

for caso in casos_de_teste:
    resposta_obtida = rag_chain.invoke(caso["pergunta"])

    resultados.append({
        "id": caso["id"],
        "pergunta": caso["pergunta"],
        "resposta_esperada": caso["resposta_esperada"],
        "resposta_obtida": resposta_obtida,
    })

    print(f"[{caso['id']}]")
    print(f"Pergunta: {caso['pergunta']}")
    print(f"Esperado: {caso['resposta_esperada']}")
    print(f"Obtido:   {resposta_obtida}")
    print("-" * 100)

print("\nTeste concluído. Resultados salvos na variável 'resultados' para análise posterior.")

Total de casos de teste: 32

[PRIV-01]
Pergunta: Qual é a legislação principal que fundamenta a política de privacidade do Voa Bank?
Esperado: A Lei Geral de Proteção de Dados (Lei nº 13.709/2018 - LGPD).
Obtido:   Não encontrou a informação.
----------------------------------------------------------------------------------------------------
[PRIV-02]
Pergunta: Por quanto tempo o Voa Bank mantém o histórico de transações dos clientes?
Esperado: 10 anos, conforme exigência do Banco Central (BACEN).
Obtido:   Não encontramos nenhuma informação sobre o período de tempo que o Voa Bank mantém o histórico de transações dos clientes.
----------------------------------------------------------------------------------------------------
[PRIV-03]
Pergunta: Se eu quiser solicitar a exclusão dos meus dados pessoais, qual o prazo de resposta e por qual canal devo pedir?
Esperado: Pelo aplicativo, em 'Configurações > Privacidade e Dados', ou pelo e-mail dpo@voabank.com.br. Prazo de até 15 dias úteis.